# QC III: Shor's Algorithm

We have now covered enough material to discuss the famous Shor's factoring algorithm. If you have not heard about this, this can be quickly summarized as providing a 
*probabilistic* factoring algorithm that runs in polynomial time. Recall that general factoring of integers is supposed to be a notoriously difficult comptuational task, so Shor's polynomial algorithm was surprising when it was first introduced (in 1994!). Needless to say, this spurred on a widespread interest in quantum computing (and piqued the interest of many cryptographers). 

Shor's algorithm consists of two main parts: the part that requires quantum computing, and the part that is purely classical processing. As fancy as this algorithm is, it might be interesting to you to hear that we've already covered the crux of the quantum computing portion of Shor's algorithm. The quantum part of Shor's algorithm is acutally fairly straightforward, and is a simple application of the quantum Fourier transform via phase estimation. Perhaps this is a testament to the ubiquity of the quantum Fourier transform. 

At the high level, Shor's algorithm can be summarized as follows. First, the quantum part requires us to convert the task of factoring an integer $N$ to a matter of 
calculating eigenvales for some unitary operator $U$ acting on some qubit system $\mathcal{H}(n)$. To do this, we will attempt to reduce the factoring problem to the *order-finding* problem: that is, for a fixed integer $a$, calculate the smallest $r$ such that $a^{r} \equiv 1$ (mod $N$). So, the algorithm begins by first randomly selecting an integer $0 < a < N$. Then, running the QPE procedure for $U$ leaves us with a qubit system in some quantum state that we can measure to obtain some classical value $v$. The measured classical value $v$ will not directly contain the answer to our order-finding problem (i.e., the integer $r$ for the fixed integer $a$), but $v$ will be a value sufficiently close to a multiple of $\frac{2^{n}}{r}$. What will happen then is that we will apply classical methods to extract the true order $r$ from the measured value $v$. 

That sounds straightforward enough -- but alas, it turns out that the order-finding problem does not *always* give us a factor of $N$. It will do so only if $r$ happens to be <b>even</b>. Furthermore, we will shortly see that $r$ being an even integer may still fail to provide us with a factor of $N$. If either $a^{r/s} + 1$ or $a^{r/2} - 1$ happens to be a multiple of $N$, then we will not be guaranteed a factor of the integer $N$.

However, of course the order $r$ depends on the randomly chosen integer $a$ from the beginning. We can just keep repeating the above steps if necessary, until we stumble upon a choice of integer $a$ for which $r$ happens to be even, and neither  $a^{r/s} + 1$ or $a^{r/2} - 1$ happen to be multiples of $N$. As long as the steps above are polynomial in time complexity, repeating the above steps a number of times until we happen upon a good choice of integer $a$ does not ultimately detract from the overall polynomial time complexity of the algorithm. 


Now let's discuss the details.

------

### Classical reduction of factoring to order-finding


Recall that the order of an integer $a$ in $(\mathbb{Z}/ N \mathbb{Z})^{\times}$ is the smallest integer $r$ such that $a^{r} \equiv 1$ modulo $N$. The important insight is that if $r$ is even, then we can write $ (a^{r/2})^{2} \equiv 1 $  modulo $N$, or $(a^{r/2})^{2} - 1 \equiv 0 \, \, (\text{mod } N)$. This gives us

$$
( a^{r/2} + 1 )(a ^{r/2} - 1 ) \equiv 0 \, \,(\text{mod } N )
$$

Thus, as long as neither $( a^{r/2} + 1 )$ or $(a ^{r/2} - 1 )$ is a multiple of $N$, then we are guaranteed that both $( a^{r/2} + 1 )$ and $(a ^{r/2} - 1 )$ have non-trivial common factors with $N$. 

Therefore, we can extract factors of $N$ via order-finding in the following way:

1. Randomly choose an integer $a$, and determine the order $r$ of $a \in (\mathbb{Z}/N \mathbb{Z})^{\times}$.

2. If $r$ is even, then use the Euclidean algorithm to effectively compute the gcd of $a^{r/2} + 1$ (or $a^{r/2} - 1$) and $N$. 

Repeat all the above steps if necessary (for example, if the randomly chosen $a$ results in an *odd* $r$)

-----------------

###  Using QPE to (approximately) solve the order-finding problem 

To understand how we can use the quantum phase estimation algorithm to solve the order-finding problem, we need to consider a unitary transformation
with eigenvalues whose values will allow us to compute orders of elements in $\mathbb{Z}/N\mathbb{Z}$. To this end, we consider the operator

$$
U_{x} \ket{y} = \ket{ xy \,  (\text{mod } N)}
$$

By direct calculation, one can see that the states defined by

$$
\ket{u_{s}} := \frac{1}{\sqrt{r}} \sum\limits_{k=0}^{r-1} \exp( \frac{ - 2\pi i s k }{r} )  \ket{ x^{k} \, (\text{mod } N)}
$$

for integers $0 \leq s \leq r - 1$ are eigenvectors of $U_{x}$, where $r$ is the order of $x$ in $(\mathbb{Z}/N\mathbb{Z})^{\times}$.
Indeed, we have that 

$$
U_{x} \ket{u_{s}} = \frac{1}{\sqrt{r}} \sum\limits_{k=0}^{r-1} \exp( \frac{ - 2\pi i s k }{r} ) \ket{ x^{k + 1} \, (\text{mod } N)}
$$

$$
= \exp( \frac{ - 2\pi i s }{r} ) \ket{u_{s}}
$$

So that $\ket{u_{s}}$ has corresponding eigenvalue $\exp( \frac{ - 2\pi i s }{r} )$. Therefore, using the QPE algorithm, we will be able to approximate $\exp( \frac{ - 2\pi i s }{r} )$ to a desired degree of precision, and then we are left with the task of extracting the value $r$ out of our approximation. 



However, we are not out of the woods yet -- recall that to perform the QPE algorithm, we need to be able to do two things: 

1. We need to be able to efficiently implement controlled $U_{x}^{2^{j}}$ gates.

2. We need to be able to prepare an eigenstate $\ket{u_{s}}$ to pass through the second register of the QPE circuit. 


We will see that while accomplishing 1. is fairly straightforward, 2. is trickier. In order to prepare the state $\ket{u_{s}}$, one must already know $r$! Fortunately, there is a workaround. The idea is that we are not necessarily interested in any individual eigenstate $\ket{u_{s}}$ -- any one of these eigenstates can be used to approximate $r$ via approximating $\exp( \frac{ - 2\pi i s }{r} )$. Therefore, we use the identity

$$
\frac{1}{\sqrt{r}} \sum\limits_{s = 0 }^{r - 1 } \ket{u_{s}} = \ket{1}_{N}
$$

which follows from the fact that $\sum \limits_{s = 1}^{r-1} e^{ \frac{2 \pi i s} { r}} = 1$. Then, the QPE circuit will act as follows:


$$
\text{QPE}_{U} \Big( \ket{0}_{2n} \ket{1}_{n} \Big) = \text{QPE}_{U} \Big( \ket{0}_{2n} (  \frac{1}{\sqrt{r}} \sum\limits_{s = 0 }^{r - 1 } \ket{u_{s}}  ) \Big)
$$
$$
= \frac{1}{\sqrt{r}} \sum\limits_{s = 0 } ^{r-1} \text{QPE}_{U} \Big( \ket{0}_{2n}  \ket{u_{s}} \Big) = \frac{1}{\sqrt{r}} \sum\limits_{s = 0}^{r-1} \ket{ 2^{2n} \frac{s}{r} } \ket{ u_{s} }
$$

Measuring the first $2n$ qubits of the resulting state will give us the QPE approximation of $\frac{s}{r}$ for some $s \in \{ 0, \cdots , r-1 \}$. 

####  Implementing controlled $U^{2^{j}}$ gates 



------------------------------


### Extracting the order $r$ from the QPE approximation